In [3]:
from transformers import PretrainedConfig, PreTrainedModel, AutoModel, AutoFeatureExtractor
from transformers.models.qwen3.configuration_qwen3 import Qwen3Config
from transformers.modeling_outputs import ModelOutput
from typing import Optional, Union
import torch
import torch.nn as nn
import os
from dataclasses import dataclass
from transformers import AutoConfig
from IPython.display import Audio, display
import os
import torch
import math
import torch.nn.functional as F
from transformers import HiggsAudioV2TokenizerModel, AutoFeatureExtractor

### Utils

In [4]:
def _gumbel_sample(logits: torch.Tensor, temperature: float) -> torch.Tensor:
    scaled_logits = logits / temperature
    u = torch.rand_like(scaled_logits)
    gumbel_noise = -torch.log(-torch.log(u + 1e-10) + 1e-10)
    return scaled_logits + gumbel_noise

def process_batch_logits(
    # Output
    batch_logits,
    
    # Variable
	B,
 	step,
	batch_input_ids,  
	schedules, 
	c_lens, 
	layer_ids, 
	tokens,
 
	# Config, Task
 	gen_config,
	task,
 
 	# Others
  	audio_mask_id,
):
    with torch.inference_mode():
        for i in range(B):
            k = schedules[i][step]
            if k <= 0:
                continue

            c_len, t_len = c_lens[i], task.target_lens[i]

            # Extract real target Logits
            # [1, C, T, V]
            c_logits = batch_logits[i : i + 1, :, c_len - t_len : c_len, :]
            u_logits = batch_logits[B + i : B + i + 1, :, :t_len, :]

            pred_tokens, scores = _predict_tokens_with_scoring(
                c_logits, u_logits, gen_config, audio_mask_id
            )

            scores = scores - (layer_ids * gen_config.layer_penalty_factor)

            if gen_config.position_temperature > 0.0:
                scores = _gumbel_sample(scores, gen_config.position_temperature)

            sample_tokens = tokens[i : i + 1, :, :t_len]
            scores.masked_fill_(
                sample_tokens != audio_mask_id, -float("inf")
            )

            _, topk_idx = torch.topk(scores.flatten(), k)
            flat_tokens = sample_tokens.flatten()
            flat_tokens[topk_idx] = pred_tokens.flatten()[topk_idx]
            sample_tokens.copy_(flat_tokens.view_as(sample_tokens))

            # Update individual slices into batched structure
            tokens[i : i + 1, :, :t_len] = sample_tokens
            batch_input_ids[i : i + 1, :, c_len - t_len : c_len] = sample_tokens
            batch_input_ids[B + i : B + i + 1, :, :t_len] = sample_tokens 
             
    return batch_input_ids, tokens

def _predict_tokens_with_scoring(c_logits, u_logits, gen_config, audio_mask_id):
	if gen_config.guidance_scale != 0:
		c_log_probs = F.log_softmax(c_logits, dim=-1)
		u_log_probs = F.log_softmax(u_logits, dim=-1)
		log_probs = torch.log_softmax(
			c_log_probs + gen_config.guidance_scale * (c_log_probs - u_log_probs),
			dim=-1,
		)
	else:
		log_probs = F.log_softmax(c_logits, dim=-1)

	log_probs[..., audio_mask_id] = -float("inf")

	if gen_config.class_temperature > 0.0:
		filtered_probs = _filter_top_k(log_probs, ratio=0.1)
		pred_tokens = _gumbel_sample(
			filtered_probs, gen_config.class_temperature
		).argmax(dim=-1)
	else:
		pred_tokens = log_probs.argmax(dim=-1)

	confidence_scores = log_probs.max(dim=-1)[0]

	return pred_tokens, confidence_scores
    

def _filter_top_k(logits: torch.Tensor, ratio: float = 0.1) -> torch.Tensor:
    k = math.ceil(ratio * logits.shape[-1])
    val, ind = logits.topk(k, dim=-1)
    probs = torch.full_like(logits, float("-inf"))
    probs.scatter_(-1, ind, val)
    return probs

### Paths

In [5]:
def _resolve_model_path(name_or_path: str) -> str:
    if os.path.isdir(name_or_path):
        return name_or_path
    from huggingface_hub import snapshot_download

    return snapshot_download(name_or_path)

In [6]:
model_path = _resolve_model_path("k2-fsa/OmniVoice")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

In [8]:
audio_tokenizer_path = model_path + "/audio_tokenizer"

### Config

In [10]:
class OmniVoiceConfig(PretrainedConfig):
    model_type = "omnivoice"
    sub_configs = {"llm_config": Qwen3Config}

    def __init__(
        self,
        audio_vocab_size: int = 1025,
        audio_mask_id: int = 1024,
        num_audio_codebook: int = 8,
        audio_codebook_weights: Optional[list[float]] = None,
        llm_config: Optional[Union[dict, Qwen3Config]] = None,
        **kwargs,
    ):
        llm_config = Qwen3Config(**llm_config)
        self.llm_config = llm_config
        
        super().__init__(**kwargs)
        self.audio_vocab_size = audio_vocab_size
        self.audio_mask_id = audio_mask_id
        self.num_audio_codebook = num_audio_codebook
        self.audio_codebook_weights = audio_codebook_weights

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [12]:
config = OmniVoiceConfig.from_pretrained(model_path)

### Models

In [14]:
audio_embeddings = nn.Embedding(
    config.num_audio_codebook * config.audio_vocab_size,
    config.llm_config.hidden_size,
)
audio_heads = nn.Linear(
    config.llm_config.hidden_size,
    config.num_audio_codebook * config.audio_vocab_size,
    bias=False,
)
llm = AutoModel.from_pretrained(
    "./omnivoice_qwen3",
    dtype=torch.float32,
)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

In [15]:
audio_embeddings.load_state_dict(
    torch.load(
        "./omnivoice_audio_embeddings/model.pt",
        map_location="cpu",
        weights_only=True,
    )
)
audio_heads.load_state_dict(
    torch.load(
        "./omnivoice_audio_heads/model.pt",
        map_location="cpu",
        weights_only=True,
    )
)

audio_embeddings = audio_embeddings.to(
    device="cuda",
    dtype=llm.dtype if hasattr(llm, "dtype") else torch.float32,
)
audio_heads = audio_heads.to(
    device="cuda",
    dtype=torch.float32,
)

llm = llm.to("cuda")

codebook_layer_offsets = torch.arange(config.num_audio_codebook) * config.audio_vocab_size
codebook_layer_offsets = codebook_layer_offsets.to("cuda")

In [16]:
audio_tokenizer = HiggsAudioV2TokenizerModel.from_pretrained(
    audio_tokenizer_path,
    torch_dtype=torch.float32,
).to(device).eval()

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

### Example Input

In [17]:
save_data = torch.load(
    "batch_voice_audio.pt",
    map_location="cuda:0",
    weights_only=False,
)

In [18]:
gen_config = save_data["gen_config"]
short_task = save_data["short_task"]
batch_input_ids = save_data["batch_input_ids"]
batch_audio_mask = save_data["batch_audio_mask"]
batch_attention_mask = save_data["batch_attention_mask"]
schedules = save_data["schedules"]
c_lens = save_data["c_lens"]
layer_ids = save_data["layer_ids"]
tokens = save_data["tokens"]

B = short_task.batch_size

In [19]:
batch_input_ids.shape, batch_audio_mask.shape, batch_attention_mask.shape

(torch.Size([2, 8, 224]), torch.Size([2, 224]), torch.Size([2, 1, 224, 224]))

### Forward Function

In [20]:
def forward(
    input_ids: torch.LongTensor,
    audio_mask: torch.Tensor,
    labels: Optional[torch.LongTensor] = None,
    attention_mask: Optional[torch.Tensor] = None,
    document_ids: Optional[torch.Tensor] = None,
    position_ids: Optional[torch.LongTensor] = None,    
):
    # prepare_embed_inputs
    input_embeddings = llm.get_input_embeddings()
    text_embeds = input_embeddings(input_ids[:, 0, :])
    shifted_ids = (
        input_ids * audio_mask.unsqueeze(1)
    ) + codebook_layer_offsets.view(1, -1, 1)   
    audio_embeds = audio_embeddings(shifted_ids).sum(dim=1)
    inputs_embeds = torch.where(
        audio_mask.unsqueeze(-1), audio_embeds, text_embeds
    )

    # llm
    llm_outputs = llm(
        inputs_embeds=inputs_embeds,
        attention_mask=attention_mask,
        return_dict=True,
        position_ids=position_ids,
    )    

    hidden_states = llm_outputs[0]

    batch_size, seq_len, _ = hidden_states.shape
    logits_flat = audio_heads(hidden_states)
    
    audio_logits = logits_flat.view(
        batch_size,
        seq_len,
        config.num_audio_codebook,
        config.audio_vocab_size,
    ).permute(0, 2, 1, 3)    
    return audio_logits

### Iterarive

In [21]:
with torch.inference_mode():
    for step in range(gen_config.num_step):
        print(f"Step - {step}")
        
        batch_logits = forward(
            input_ids=batch_input_ids,
            audio_mask=batch_audio_mask,
            attention_mask=batch_attention_mask,
        ).to(torch.float32)  

        batch_input_ids, tokens = process_batch_logits(
            # Output
            batch_logits,
            
            # Variable
        	B,
         	step,
        	batch_input_ids,  
        	schedules, 
        	c_lens, 
        	layer_ids, 
        	tokens,
         
        	# Model, Config, Task
         	gen_config,
        	short_task,

            # Others
            config.audio_mask_id
        )        

Step - 0
Step - 1
Step - 2
Step - 3
Step - 4
Step - 5
Step - 6
Step - 7
Step - 8
Step - 9
Step - 10
Step - 11
Step - 12
Step - 13
Step - 14
Step - 15
Step - 16
Step - 17
Step - 18
Step - 19
Step - 20
Step - 21
Step - 22
Step - 23
Step - 24
Step - 25
Step - 26
Step - 27
Step - 28
Step - 29
Step - 30
Step - 31


### Decode

In [22]:
output = [tokens[i, :, : short_task.target_lens[i]] for i in range(B)]
data = output[0].unsqueeze(0).to(llm.device)

In [23]:
with torch.inference_mode():
    decoded_audio = audio_tokenizer.decode(data)

In [24]:
decoded_ref_wav = decoded_audio.audio_values[0][0].cpu().numpy()
display(Audio(decoded_ref_wav, rate=24000))